# 04_BL_Walkforward — Black-Litterman 실험 실행

**역할**: bl_config.py에 정의된 모든 실험을 walk-forward로 실행하고 결과를 `results/` 폴더에 저장.

## 실행 순서
1. 데이터 로드 (panel, daily_ret)
2. LSTM 예측 로드 (`bl_runner.load_lstm_pred`)
3. monthly_cache 빌드 (`bl_runner.build_monthly_cache`) — 모든 슬롯 공유
4. 전체 실험 walk_forward 실행 → pkl 저장

> Dispatcher (`get_vol_series`, `get_prior_weights`, `get_Q`, `get_omega`) 와 `walk_forward` 본체는 [`bl_runner.py`](bl_runner.py) 에 정의됨.
> 분석/시각화는 **05b_Analyze.ipynb** 에서.

---

**관련 가이드**:
- [PROJECT_OVERVIEW.md](docs/PROJECT_OVERVIEW.md) (전체 파이프라인)
- [BL_EXPERIMENT_GUIDE.md](docs/BL_EXPERIMENT_GUIDE.md) — 슬롯 정의·수식·실행법


In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')

warnings.filterwarnings('ignore')

# BL 백테스트 엔진 (dispatcher + walk_forward 본체)
from bl_runner import load_pred_csv, build_monthly_cache, walk_forward
from bl_config import EXPERIMENTS as EXPERIMENTS_LSTM, BASELINE
from bl_config_ann import EXPERIMENTS as EXPERIMENTS_ANN

EXPERIMENTS = EXPERIMENTS_LSTM + EXPERIMENTS_ANN

BASE_DIR    = Path.cwd()
DATA_DIR    = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

assert DATA_DIR.exists(), f'data/ 폴더 없음. 01_DataCollection.ipynb 먼저 실행하세요.\n경로: {DATA_DIR}'

# ── 공통 파라미터 ──────────────────────────────────────────────
TRAIN_WINDOW  = 60
THRESH_DAILY  = 0.9
TAU           = 0.1
PCT_GROUP     = 0.30
START_PRED    = '2010-01-01'

print(f'설정 완료')
print(f'  DATA_DIR    : {DATA_DIR}')
print(f'  RESULTS_DIR : {RESULTS_DIR}')
print(f'  실험 수 (LSTM): {len(EXPERIMENTS_LSTM)}')
print(f'  실험 수 (ANN) : {len(EXPERIMENTS_ANN)}')
print(f'  실험 수 (TOT) : {len(EXPERIMENTS)}')


설정 완료
  DATA_DIR    : c:\Users\서윤범\Desktop\temp\finance_project\final_pt\data
  RESULTS_DIR : c:\Users\서윤범\Desktop\temp\finance_project\final_pt\results
  실험 수 (LSTM): 108
  실험 수 (ANN) : 108
  실험 수 (TOT) : 216


In [2]:
# ── 데이터 로드 ───────────────────────────────────────────────
panel     = pd.read_csv(DATA_DIR / 'monthly_panel.csv', parse_dates=['date'])
panel     = panel.set_index(['date', 'ticker'])
daily_ret = pd.read_pickle(DATA_DIR / 'daily_returns.pkl')

all_dates  = panel.index.get_level_values('date').unique().sort_values()
pred_dates = all_dates[all_dates >= START_PRED]

spy_series = panel['spy_ret'].groupby(level='date').first()
rf_series  = panel['rf_1m'].groupby(level='date').first()

# q_mode='ff3_paper_mean' 슬롯이 사용할 보조 입력
# - ret_pivot: 월별 종목 수익률 매트릭스 (FF3 회귀 의존변수)
# - ff3      : Fama-French 3-factor 월별 시계열
ret_pivot = panel['ret_1m'].unstack('ticker')
ff3_path  = DATA_DIR / 'ff3_monthly.csv'
assert ff3_path.exists(), f'FF3 월별 파일 없음: {ff3_path}'
ff3 = pd.read_csv(ff3_path, index_col=0, parse_dates=True)
# rf 컬럼은 panel rf_1m 로 별도 처리. mkt_rf/smb/hml 만 필요.
_need_cols = {'mkt_rf', 'smb', 'hml'}
assert _need_cols.issubset(set(ff3.columns)), (
    f'ff3_monthly.csv 컬럼 부족: {set(ff3.columns)} (필요: {_need_cols})')

print(f'패널       : {panel.shape}')
print(f'일별 수익률: {daily_ret.shape}')
print(f'ret_pivot  : {ret_pivot.shape}  (월별 × 종목 매트릭스)')
print(f'ff3        : {ff3.shape}        ({ff3.index.min().date()} ~ {ff3.index.max().date()})')
print(f'예측 기간  : {pred_dates[0].date()} ~ {pred_dates[-1].date()} ({len(pred_dates)}개월)')


패널       : (103878, 13)
일별 수익률: (5595, 824)
ret_pivot  : (252, 617)  (월별 × 종목 매트릭스)
ff3        : (1197, 4)        (1926-07-31 ~ 2026-03-31)
예측 기간  : 2010-01-31 ~ 2025-12-31 (192개월)


In [3]:
# ── LSTM/ANN 예측값 로드 (없으면 None) ────────────────────────────
# LSTM: 03b_Volatility_Forecasting.ipynb 산출물
# ANN : _dev/_train_paper_ann.py 산출물 (Pyo & Lee 2018 baseline)
_lstm_path = Path(BASELINE.get('lstm_pred_path', ''))
_ann_path  = DATA_DIR / 'paper_ann_predictions.csv'

lstm_state = load_pred_csv(_lstm_path, pred_dates)
ann_state  = load_pred_csv(_ann_path,  pred_dates)

LSTM_AVAILABLE = lstm_state['available']
ANN_AVAILABLE  = ann_state['available']

if LSTM_AVAILABLE:
    print(f'LSTM 예측 로드: {len(lstm_state["preds"]):,}행')
    print(f'  월별 피벗   : {lstm_state["monthly"].shape[0]}개월 × {lstm_state["monthly"].shape[1]}종목')
else:
    print(f'⚠  LSTM 예측 파일 없음 → LSTM 슬롯 모두 스킵')
    print(f'   경로: {_lstm_path}')

if ANN_AVAILABLE:
    print(f'ANN  예측 로드: {len(ann_state["preds"]):,}행')
    print(f'  월별 피벗   : {ann_state["monthly"].shape[0]}개월 × {ann_state["monthly"].shape[1]}종목')
else:
    print(f'⚠  ANN 예측 파일 없음 → ANN 슬롯 모두 스킵')
    print(f'   경로: {_ann_path}')
    print(f'   _dev/_train_paper_ann.py 를 먼저 실행해서 생성하세요.')


LSTM 예측 로드: 2,498,243행
  월별 피벗   : 192개월 × 614종목
ANN  예측 로드: 103,379행
  월별 피벗   : 192개월 × 598종목


## Dispatcher 함수

Dispatcher (`get_vol_series`, `get_prior_weights`, `get_Q`, `get_omega`) 와 `walk_forward` 본체는 **[`bl_runner.py`](bl_runner.py)** 에 정의됨.

본 노트북은 import 만 하고, 05b_Analyze.ipynb 의 sensitivity sweep 도 같은 모듈을 재사용.


In [4]:
# ── 월별 공유 데이터 캐시 빌드 (Sigma, π 컴포넌트) ───────────────
# 모든 실험에서 동일하므로 1회만 계산. 실험별로 달라지는 것:
#   vol_series(p_mode), P(p_weight), Q, omega, w → walk_forward 안에서 처리.
monthly_cache = build_monthly_cache(
    panel=panel,
    daily_ret=daily_ret,
    pred_dates=pred_dates,
    all_dates=all_dates,
    spy_series=spy_series,
    rf_series=rf_series,
    train_window=TRAIN_WINDOW,
    thresh_daily=THRESH_DAILY,
    verbose=True,
)
print('이후 실험은 Sigma 재계산 없이 캐시에서 로드')

  캐시 2010-01 (1/192, 1%) | 경과 0.0분 | ETA 0.0분
  캐시 2012-01 (25/192, 13%) | 경과 0.0분 | ETA 0.3분
  캐시 2014-01 (49/192, 26%) | 경과 0.1분 | ETA 0.3분
  캐시 2016-01 (73/192, 38%) | 경과 0.1분 | ETA 0.2분
  캐시 2018-01 (97/192, 51%) | 경과 0.2분 | ETA 0.2분
  캐시 2020-01 (121/192, 63%) | 경과 0.3분 | ETA 0.2분
  캐시 2022-01 (145/192, 76%) | 경과 0.3분 | ETA 0.1분
  캐시 2024-01 (169/192, 88%) | 경과 0.4분 | ETA 0.1분

캐시 완료: 192/192개월 | 0.5분
이후 실험은 Sigma 재계산 없이 캐시에서 로드


## walk_forward 실행

`bl_runner.walk_forward(cfg, monthly_cache, pred_dates, lstm_state, spy_series, ...)` 를 호출하여 슬롯별 백테스트.

결과 dict: `config`, `ret` (net), `gross_ret`, `spy_ret`, `weights`, `comp`, `meta`, `errors`.


In [5]:
# ══════════════════════════════════════════════════════════════
# 전체 실험 실행 + 저장 (LSTM + ANN 슬롯 통합)
# ══════════════════════════════════════════════════════════════
import time
SKIP_IF_EXISTS = True   # True → 이미 저장된 실험은 재실행 않고 스킵

completed, skipped = [], []
_all_t0 = time.time()

# p_mode 별 가용성 게이팅
def _can_run(cfg):
    p_mode = cfg.get('p_mode', 'lstm_predicted')
    if p_mode == 'lstm_predicted':
        return LSTM_AVAILABLE
    if p_mode == 'ann_predicted':
        return ANN_AVAILABLE
    return False

run_list = [
    cfg for cfg in EXPERIMENTS
    if _can_run(cfg)
    and not (SKIP_IF_EXISTS and (RESULTS_DIR / f"{cfg['name']}.pkl").exists())
]
skipped = [
    cfg['name'] for cfg in EXPERIMENTS
    if not _can_run(cfg)
    or (SKIP_IF_EXISTS and (RESULTS_DIR / f"{cfg['name']}.pkl").exists())
]

print(f'실행 예정      : {len(run_list)}개 / 전체 {len(EXPERIMENTS)}개')
print(f'  - LSTM 슬롯  : {sum(1 for c in run_list if c.get("p_mode") == "lstm_predicted")}')
print(f'  - ANN  슬롯  : {sum(1 for c in run_list if c.get("p_mode") == "ann_predicted")}')
print(f'스킵 (기존/미가용): {len(skipped)}개\n')

for cfg in run_list:
    name     = cfg['name']
    out_path = RESULTS_DIR / f'{name}.pkl'

    done_so_far = len(completed)
    remaining   = len(run_list) - done_so_far
    elapsed_all = time.time() - _all_t0
    avg_per_exp = elapsed_all / done_so_far if done_so_far > 0 else 0
    eta_all     = avg_per_exp * remaining

    print(f'\n[{done_so_far+1}/{len(run_list)}] {name}  |  '
          f'전체 경과 {elapsed_all/60:.1f}분  |  ETA {eta_all/60:.1f}분')

    result = walk_forward(
        cfg, monthly_cache, pred_dates, lstm_state,
        spy_series=spy_series, tau=TAU, pct_group=PCT_GROUP, verbose=True,
        ann_state=ann_state,
        ret_pivot=ret_pivot, ff3=ff3, rf_series=rf_series,
    )

    with open(out_path, 'wb') as f:
        pickle.dump(result, f)

    completed.append(name)

total_elapsed = (time.time() - _all_t0) / 60
print(f'\n{"=" * 60}')
print(f'완료: {len(completed)}개 / 스킵: {len(skipped)}개 / 총 {total_elapsed:.1f}분')
print(f'저장 위치: {RESULTS_DIR}')


실행 예정      : 216개 / 전체 216개
  - LSTM 슬롯  : 108
  - ANN  슬롯  : 108
스킵 (기존/미가용): 0개


[1/216] mat_mcap_mcap_fix_he  |  전체 경과 0.0분  |  ETA 0.0분
  [mat_mcap_mcap_fix_he] 2010-01 (1/192, 1%) | 경과 0.0분 | ETA 0.0분
  [mat_mcap_mcap_fix_he] 2011-01 (13/192, 7%) | 경과 0.5분 | ETA 6.6분
  [mat_mcap_mcap_fix_he] 2012-01 (25/192, 13%) | 경과 0.8분 | ETA 5.4분
  [mat_mcap_mcap_fix_he] 2013-01 (37/192, 19%) | 경과 1.1분 | ETA 4.7분
  [mat_mcap_mcap_fix_he] 2014-01 (49/192, 26%) | 경과 1.7분 | ETA 5.1분
  [mat_mcap_mcap_fix_he] 2015-01 (61/192, 32%) | 경과 2.3분 | ETA 5.0분
  [mat_mcap_mcap_fix_he] 2016-01 (73/192, 38%) | 경과 2.8분 | ETA 4.5분
  [mat_mcap_mcap_fix_he] 2017-01 (85/192, 44%) | 경과 3.3분 | ETA 4.1분
  [mat_mcap_mcap_fix_he] 2018-01 (97/192, 51%) | 경과 4.0분 | ETA 3.9분
  [mat_mcap_mcap_fix_he] 2019-01 (109/192, 57%) | 경과 4.6분 | ETA 3.5분
  [mat_mcap_mcap_fix_he] 2020-01 (121/192, 63%) | 경과 5.4분 | ETA 3.1분
  [mat_mcap_mcap_fix_he] 2021-01 (133/192, 69%) | 경과 6.0분 | ETA 2.7분
  [mat_mcap_mcap_fix_he] 2022-01 (145/192, 

KeyboardInterrupt: 